# Regression and genuine held-out evaluation (models 0, A–D, RF)

All model definitions, splits and metrics come from `src/model_comparison.py` (CC-9); this notebook only runs them, so there are no parallel model specs. Target: ground speed in **km/h** (tables also show m/s). Altitude enters as a piecewise-linear term with a knot at `altitude_knot_m` (config.json, 3,000 m; `src/features.py`), zone (reference: far) with zone × altitude interactions.

| Model | Inputs |
|---|---|
| 0 | mean of the training speeds |
| A | altitude knot (`alt_low_km`, `alt_high_km`) |
| B | A + zone, zone × altitude, airport, `distance_km`, flight phase (control) |
| C | B + aircraft category (`Unbekannt` kept as its own level) |
| D | C + modelled tail- and crosswind (only when wind was joined) |
| RF | random forest on B's inputs plus latitude/longitude (comparison only) |

Fixed specifications, no tuning on test data; every imputer, scaler and encoder is fitted on the training rows only. These models estimate simultaneous ground speed, not future trajectories. The later-days split is the main result (hypothesis 2: does R² rise from A to D?), the held-out-aircraft split tests transfer to new aircraft. The 7-date guard stays: nothing runs on fewer than seven observed UTC dates.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import model_comparison as mc
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
config = json.loads((ROOT / "config.json").read_text())
# One row per observation: aircraft category from observations_enriched, wind (model D) from observation_wind.
with closing(sqlite3.connect(DB)) as db:
    n_observations = db.execute("SELECT COUNT(*) FROM observations").fetchone()[0]
    has_wind_table = db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name = 'observation_wind'").fetchone()[0]
    if has_wind_table:
        query = ("SELECT e.*, w.tailwind_ms, w.crosswind_ms, w.wind_matched FROM observations_enriched e "
                 "LEFT JOIN observation_wind w ON e.icao24 = w.icao24 AND e.time_position = w.time_position")
    else:
        query = "SELECT * FROM observations_enriched"
    data = pd.read_sql_query(query, db)
assert len(data) == n_observations, "The joins changed the number of observations."
assert not data[["icao24", "time_position"]].duplicated().any()
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
print("Observations:", len(data), "| with wind:", int(data.get("wind_matched", pd.Series(dtype=float)).fillna(0).sum()))

In [ ]:
# Model frame (near/middle/far, knot features, interactions) and the 7-date guard.
work = mc.model_frame(data, config)
specs = mc.model_specs(config, include_wind=mc.has_wind(work))
for name, spec in specs.items():
    print(f"{name}: {spec.features}")
train, test = mc.split_by_date(work)
print("Training dates:", sorted(train.date_utc.unique()), "Test dates:", sorted(test.date_utc.unique()))
print("Shared aircraft in the temporal split:", len(set(train.icao24) & set(test.icao24)))
display(pd.concat([train.groupby("zone").size().rename("train"), test.groupby("zone").size().rename("test")], axis=1))

In [ ]:
metrics_days, predictions_days, fitted_days = mc.evaluate_models(train, test, "later_days", config)
display(metrics_days.round(3))
# Coefficients of the linear models, fitted on the training days: per training SD and in original units.
coefficients = mc.coefficient_table(fitted_days, train, config)
display(coefficients.round(3))
coefficients.to_csv(OUT / "tables/model_coefficients.csv", index=False)
predictions_days.to_csv(OUT / "tables/temporal_predictions.csv", index=False)
# Diagnostics for the prespecified model B, not used to choose a model.
model_b = next(name for name in specs if name.startswith("B"))
predictions_days["residual_kmh"] = predictions_days.speed_kmh - predictions_days[model_b]
sample = predictions_days.sample(min(3000, len(predictions_days)), random_state=42)
sns.scatterplot(data=sample, x="altitude_m", y="residual_kmh", hue="zone", hue_order=["near", "middle", "far"],
                palette={"near": "#DD8452", "middle": "#55A868", "far": "#4C72B0"}, alpha=.4)
plt.axhline(0, color="black", lw=1); plt.xlabel("Altitude (m; geo, else baro)"); plt.ylabel("Observed minus predicted, model B (km/h)")
plt.tight_layout(); plt.savefig(OUT / "figures/residuals_by_altitude.png", dpi=160); plt.show()
display(predictions_days.groupby(["airport", "zone"]).residual_kmh.agg(["count", "median", "std"]).round(1))

In [ ]:
train_aircraft, test_aircraft = mc.split_by_aircraft(work)
metrics_aircraft, predictions_aircraft, _ = mc.evaluate_models(train_aircraft, test_aircraft, "other_aircraft", config)
display(metrics_aircraft.round(3))

## Leakage demonstration (not a result)

The next cell uses the **same features** with a **random row split** (`random_state=42`). Positions of the same aircraft, often seconds apart on the same flight, then end up in training and test. A flexible model (the random forest with latitude/longitude) can recognise the flight and «predict» its speed from neighbouring points; its error looks much smaller than it would be for a new day or a new aircraft. This is shown only to explain why we split by day and by aircraft. **Demonstration, nicht Ergebnis**: these numbers are never reported as model performance.

In [ ]:
train_random, test_random = mc.split_random_demo(work)
metrics_random, predictions_random, _ = mc.evaluate_models(train_random, test_random, "random_rows_demo", config)
display(metrics_random.round(3))
print("DEMONSTRATION ONLY - not a result.")

In [ ]:
metrics = pd.concat([metrics_days, metrics_aircraft, metrics_random], ignore_index=True)
metrics.to_csv(OUT / "tables/model_metrics.csv", index=False)
# 95 % bootstrap interval of the RMSE, resampling whole test aircraft (repeated positions are not independent).
model_names = list(fitted_days)
intervals = pd.concat([mc.rmse_bootstrap(predictions_days, model_names, "later_days"),
                       mc.rmse_bootstrap(predictions_aircraft, model_names, "other_aircraft"),
                       mc.rmse_bootstrap(predictions_random, model_names, "random_rows_demo")], ignore_index=True)
intervals.to_csv(OUT / "tables/model_rmse_bootstrap.csv", index=False)
display(metrics.pivot_table(index="model", columns=["split", "group"], values="RMSE_kmh").round(1))
display(metrics.loc[metrics.group.eq("all")].pivot_table(index="model", columns="split", values="R_squared").round(3))
mc.rmse_figure(metrics, OUT / "figures/model_rmse_by_zone_split.png", intervals)
from IPython.display import Image
display(Image(OUT / "figures/model_rmse_by_zone_split.png"))

## Interpretation (group)

*Placeholder.* How to read the results:

- **Hypothesis 2** (slide 11): R² and RMSE of the later-days split for A → B → C → D. Which step brings the largest improvement, and is it larger than the bootstrap interval?
- **Transfer:** does the held-out-aircraft split tell the same story? A model that is good on later days but not on new aircraft learned the aircraft mix.
- **RMSE per zone, not R² per zone:** R² depends on how much the speeds vary inside a group. Near the airports speeds range from approach to climb-out, en route they are close to cruise speed; the same absolute error gives very different R². Compare zones only with RMSE/MAE in km/h.
- **Leakage demonstration:** the random split looks better, especially for the random forest. Explain why (same flights on both sides) and why it is not reported.
- Coefficients (`model_coefficients.csv`): `alt_low_km` / `alt_high_km` in km/h per 1,000 m below/above the knot, `*_x_near` the extra slope near the airports. The formal test with p-values and cluster-robust intervals is in notebook 03.